# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

Note: My name (Parham Rahimi) was not in the credentials file. I am using the first entry (Nazanin Hesari) as instructed.

In [2]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

MLFLOW_USERNAME = "nazanin_hesari@qbc12.local"
MLFLOW_PASSWORD = "your_mlflow_password"
EXPERIMENT_NAME = "qbc12_hw02_student_nazanin_hesari"

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    print("Connected to remote MLflow server.")
except Exception as e:
    print(f"Remote unavailable, using local SQLite tracking.")
    local_db = Path.cwd() / "outputs/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{local_db}")
    mlflow.set_experiment(EXPERIMENT_NAME)
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)


Remote unavailable, using local SQLite tracking.


MLflow tracking URI: sqlite:////Users/parhamrahimi/Documents/bluBank/Quera/HW2/B/outputs/mlflow.db
Experiment: qbc12_hw02_student_nazanin_hesari
Experiment ID: 1


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_student.csv
data/features/listing_availability_features_v1_student.parquet
data/features/listing_availability_features_v1_student_metadata.json
```

In [3]:
DATASET_VERSION = "v1_student"

# Resolve the project root from the notebook location
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "data" / "features"

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

print("Looking for data in:", FEATURE_DIR)

# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
    print("Loaded Parquet:", parquet_path)
elif csv_path.exists():
    feature_df = pd.read_csv(csv_path)
    print("Loaded CSV:", csv_path)
else:
    raise FileNotFoundError(f"Neither {parquet_path} nor {csv_path} found")

# Load metadata if it exists.
metadata = {}
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)

# Remove any duplicate columns
feature_df = feature_df.loc[:, ~feature_df.columns.duplicated()]
print(feature_df.shape)
feature_df.head()


Looking for data in: /Users/parhamrahimi/Documents/bluBank/Quera/HW2/B/data/features


Loaded Parquet: /Users/parhamrahimi/Documents/bluBank/Quera/HW2/B/data/features/listing_availability_features_v1_student.parquet
(10480, 33)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,property_type,room_type,accommodates,bathrooms,bedrooms,...,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,cutoff_date,dataset_version
0,1443670960781261954,30,30,1.0,0,Entire rental unit,Entire home/apt,2,1.0,1.0,...,91,1.0,2.0,30.0,31,1.0,2.0,30.0,2026-08-11,v1_student
1,896043282611946316,30,0,0.0,1,Entire rental unit,Entire home/apt,2,1.5,1.0,...,0,0.0,5.0,25.0,0,0.0,5.0,25.0,2026-08-11,v1_student
2,958726726744532841,30,0,0.0,1,Entire condo,Entire home/apt,2,1.0,1.0,...,0,0.0,2.0,7.0,0,0.0,2.0,7.0,2026-08-11,v1_student
3,39969190,30,0,0.0,1,Entire rental unit,Entire home/apt,2,1.5,1.0,...,0,0.0,3.0,9.0,0,0.0,3.0,9.0,2026-08-11,v1_student
4,1093563123501570178,30,0,0.0,1,Entire condo,Entire home/apt,2,1.0,1.0,...,0,0.0,3.0,30.0,0,0.0,3.0,30.0,2026-08-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

assert TARGET_COL in feature_df.columns, "Target column not found"

y = feature_df[TARGET_COL].copy()

clean_feature_cols = [c for c in feature_df.columns if c not in FORBIDDEN_MODEL_COLUMNS]
X_clean = feature_df[clean_feature_cols].copy()
X_clean = X_clean.loc[:, ~X_clean.columns.duplicated()]
clean_feature_cols = list(X_clean.columns)

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())
print()
print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64

Clean feature count: 26
['property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'neighbourhood_name', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]

# Make sure we didn't accidentally include the target
assert TARGET_COL not in leaky_feature_cols, "Target column in leaky features!"

X_leaky = feature_df[leaky_feature_cols].copy()
X_leaky = X_leaky.loc[:, ~X_leaky.columns.duplicated()]
leaky_feature_cols = list(X_leaky.columns)

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Also split the leaky set for the leaky run
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [7]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# Identify numeric and categorical columns from X_clean
numeric_cols = X_clean.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_cols = [c for c in X_clean.columns if c not in numeric_cols]

# Also exclude boolean columns from numeric - treat them as categorical
bool_cols = X_clean.select_dtypes(include=["bool"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in bool_cols]
for bc in bool_cols:
    if bc not in categorical_cols:
        categorical_cols.append(bc)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols), numeric_cols)
print("Categorical columns:", len(categorical_cols), categorical_cols)

Numeric columns: 21 ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'listing_price', 'minimum_nights', 'maximum_nights', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']
Categorical columns: 5 ['property_type', 'room_type', 'instant_bookable', 'is_superhost', 'neighbourhood_name']


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [8]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    y_score = get_positive_scores(model, X_test)
    y_pred = (y_score >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
    }

    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [9]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    run_dir = ARTIFACT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # Confusion matrix plot
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix - {run_name}")
    cm_path = run_dir / "confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()

    # Classification report
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    with open(run_dir / "classification_report.json", "w") as f:
        json.dump(report, f, indent=2)

    # Feature columns
    with open(run_dir / "feature_columns.json", "w") as f:
        json.dump(feature_cols, f, indent=2)

    # Dataset metadata snapshot
    with open(run_dir / "dataset_metadata_snapshot.json", "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    return run_dir

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [10]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    """Fit, evaluate, and log a complete MLflow run."""
    with mlflow.start_run(run_name=run_name):
        # Fit the pipeline
        pipeline.fit(X_train, y_train)

        # Evaluate
        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline, X_test, y_test, threshold=threshold
        )

        # Log params
        mlflow.log_params(model_params)
        mlflow.log_param("threshold", threshold)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log tags
        mlflow.set_tags(tags)
        mlflow.set_tag("dataset_version", DATASET_VERSION)

        # Save and log artifacts
        artifact_dir = save_run_artifacts(
            run_name, y_test.values, y_pred, feature_cols, metadata
        )
        mlflow.log_artifacts(str(artifact_dir))

        # Log the sklearn Pipeline model
        mlflow.sklearn.log_model(pipeline, "model", serialization_format="pickle")

        print(f"Run '{run_name}' - F1: {metrics['f1']:.4f}, AUC: {metrics['roc_auc']:.4f}")

        return metrics

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [11]:
# Build preprocessor for leaky features (same column type detection)
leaky_numeric = X_train_leaky.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
leaky_cat = [c for c in X_train_leaky.columns if c not in leaky_numeric]

leaky_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, leaky_numeric),
        ("categorical", categorical_transformer, leaky_cat),
    ],
    remainder="drop",
)

pipeline_v0 = Pipeline([
    ("preprocessor", leaky_preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=pipeline_v0,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    feature_cols=leaky_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 2000},
    tags={
        "leakage_status": "leaky",
        "known_defect": "uses future_available_rate_30d",
        "model_family": "logistic_regression",
    },
)

2026/07/07 18:57:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/07 18:57:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'v0_leaky_logistic_regression' - F1: 0.9994, AUC: 1.0000


{'accuracy': 0.9990458015267175,
 'precision': 0.9993746091307066,
 'recall': 0.9993746091307066,
 'f1': 0.9993746091307066,
 'roc_auc': 0.9999962250048131}

## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [12]:
pipeline_v1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=pipeline_v1,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "DummyClassifier", "strategy": "most_frequent"},
    tags={
        "leakage_status": "clean",
        "model_family": "dummy",
    },
)

2026/07/07 18:57:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/07 18:57:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'v1_dummy_baseline' - F1: 0.8655, AUC: 0.5000


{'accuracy': 0.762881679389313,
 'precision': 0.762881679389313,
 'recall': 1.0,
 'f1': 0.8654939106901218,
 'roc_auc': 0.5}

## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [13]:
pipeline_v2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=pipeline_v2,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 2000, "class_weight": "None"},
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
    },
)

2026/07/07 18:57:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/07 18:57:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'v2_clean_logistic_regression' - F1: 0.9790, AUC: 0.9831


{'accuracy': 0.9675572519083969,
 'precision': 0.967623701893708,
 'recall': 0.9906191369606003,
 'f1': 0.9789864029666254,
 'roc_auc': 0.9831459048223046}

## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [14]:
pipeline_v3 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"
    )),
])

run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=pipeline_v3,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 2000, "class_weight": "balanced"},
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
    },
)

2026/07/07 18:57:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/07 18:57:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'v3_balanced_logistic_regression' - F1: 0.9826, AUC: 0.9844


{'accuracy': 0.9732824427480916,
 'precision': 0.9759407772979642,
 'recall': 0.9893683552220137,
 'f1': 0.9826086956521739,
 'roc_auc': 0.984369003262854}

## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [15]:
# Fit the model once, then evaluate at multiple thresholds
thresh_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
thresh_pipeline.fit(X_train, y_train)

thresholds = [0.30, 0.40, 0.50, 0.60]

for t in thresholds:
    run_name = f"v4_threshold_{str(t).replace('.', 'p')}"
    with mlflow.start_run(run_name=run_name):
        metrics, y_pred, y_score = evaluate_binary_classifier(
            thresh_pipeline, X_test, y_test, threshold=t
        )
        mlflow.log_params({"model": "LogisticRegression", "max_iter": 2000})
        mlflow.log_param("threshold", t)
        mlflow.log_metrics(metrics)
        mlflow.set_tags({
            "leakage_status": "clean",
            "model_family": "logistic_regression",
        })
        mlflow.set_tag("dataset_version", DATASET_VERSION)
        print(f"  Threshold {t}: F1={metrics['f1']:.4f}, Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}")

  Threshold 0.3: F1=0.9775, Prec=0.9641, Rec=0.9912
  Threshold 0.4: F1=0.9778, Prec=0.9653, Rec=0.9906
  Threshold 0.5: F1=0.9790, Prec=0.9676, Rec=0.9906
  Threshold 0.6: F1=0.9805, Prec=0.9706, Rec=0.9906


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [16]:
rf_params = {
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
}

pipeline_v5 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(**rf_params)),
])

run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=pipeline_v5,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "RandomForestClassifier", **{k: str(v) for k, v in rf_params.items()}},
    tags={
        "leakage_status": "clean",
        "model_family": "random_forest",
    },
)

2026/07/07 18:57:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/07 18:57:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'v5_random_forest' - F1: 0.9869, AUC: 0.9909


{'accuracy': 0.9799618320610687,
 'precision': 0.9856519026824704,
 'recall': 0.9881175734834271,
 'f1': 0.9868831980012492,
 'roc_auc': 0.9909450448784011}

## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [17]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

comparison_cols = [
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc",
]

available_cols = [c for c in comparison_cols if c in runs_df.columns]
comparison_df = runs_df[available_cols].copy()
comparison_df.columns = [c.replace("tags.", "").replace("metrics.", "") for c in available_cols]
comparison_df = comparison_df.round(4)

comparison_df

,mlflow.runName,leakage_status,model_family,accuracy,precision,recall,f1,roc_auc
0,v0_leaky_logistic_regression,leaky,logistic_regression,0.9990,0.9994,0.9994,0.9994,1.0000
1,v5_random_forest,clean,random_forest,0.9800,0.9857,0.9881,0.9869,0.9909
2,v3_balanced_logistic_regression,clean,logistic_regression,0.9733,0.9759,0.9894,0.9826,0.9844
3,v4_threshold_0p6,clean,logistic_regression,0.9699,0.9706,0.9906,0.9805,0.9831
4,v4_threshold_0p5,clean,logistic_regression,0.9676,0.9676,0.9906,0.9790,0.9831
5,v2_clean_logistic_regression,clean,logistic_regression,0.9676,0.9676,0.9906,0.9790,0.9831
6,v4_threshold_0p4,clean,logistic_regression,0.9656,0.9653,0.9906,0.9778,0.9831
7,v4_threshold_0p3,clean,logistic_regression,0.9652,0.9641,0.9912,0.9775,0.9831
8,v1_dummy_baseline,clean,dummy,0.7629,0.7629,1.0000,0.8655,0.5000


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [18]:
# Pick the best clean run by F1 score
runs_detail = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

# Filter to only clean runs
clean_runs = runs_detail[runs_detail.get("tags.leakage_status") == "clean"]
if len(clean_runs) == 0:
    clean_runs = runs_detail[~runs_detail.get("tags.leakage_status", "").isin(["leaky"])]

if len(clean_runs) > 0:
    best_run = clean_runs.iloc[0]
    BEST_RUN_ID = best_run["run_id"]
else:
    BEST_RUN_ID = runs_detail.iloc[0]["run_id"]

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

best_run_name = client.get_run(BEST_RUN_ID).data.tags.get("mlflow.runName", "unknown")
print("Selected best run:", BEST_RUN_ID)
print("Run name:", best_run_name)

Selected best run: 41f64c3b8cf2447a88a5ff6a8694a244
Run name: v5_random_forest


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [19]:
final_explanation = """
I selected the Random Forest model (v5_random_forest) as the final candidate because it had the highest F1 score among clean models while maintaining a good balance between precision and recall. The leaky model (v0) was rejected because it used future_available_rate_30d, which is information from the label window and would not be available at prediction time in production. The class-weighted logistic regression (v3) improved recall over the baseline LR but had lower precision. Next, I would try hyperparameter tuning for the Random Forest (e.g., grid search over n_estimators and max_depth) and also try gradient boosting models like XGBoost.
"""

print(final_explanation)


I selected the Random Forest model (v5_random_forest) as the final candidate because it had the highest F1 score among clean models while maintaining a good balance between precision and recall. The leaky model (v0) was rejected because it used future_available_rate_30d, which is information from the label window and would not be available at prediction time in production. The class-weighted logistic regression (v3) improved recall over the baseline LR but had lower precision. Next, I would try hyperparameter tuning for the Random Forest (e.g., grid search over n_estimators and max_depth) and also try gradient boosting models like XGBoost.

